# Apply your R skills to create a sample description

Open file and report all variable names. If column names are listed, data is imported correctly.

In [38]:
df = read.csv("ihsm26_StudentLifeData.isle.csv", header = TRUE, stringsAsFactors = FALSE, fileEncoding = "UTF-8-BOM")
names(df)

 [1] "student" "Z1"      "H1"      "H2"      "H3"      "H4"      "R1"     
 [8] "R2"      "R3"      "R4"      "DMU1"    "DMU2"    "DMU3_A"  "DMU3_B" 
[15] "DMU3_C"  "DMU3_D"  "DMU3_E"  "DMU3_F"  "DMU3_G"  "DMU3_H"  "DMU3_I" 
[22] "DMU3_J"  "DMU4"    "DH1"     "DH2_A"   "DH2_B"   "DH2_C"   "DH2_D"  
[29] "DH2_E"   "DH3"     "DH4"     "S1"      "S2"      "S3"      "S4"     
[36] "L1"      "L2"      "L3"      "L4"      "L5"      "G1"      "G2"     
[43] "G3"      "SQ1"     "SQ2"     "SQ3"     "SQ4"     "SQ5"     "SQ6"    
[50] "SQ7"     "SQ8"     "SQ9"    

How many rows (cases) does the dataset include?

In [3]:
nrow(df)

[1] 457

## Real world challenges

Statistical functions often return NA due to missing values.

In [4]:
mean(df$SQ1)

[1] NA

In many cases you can tell R to remoce the NA before calculation (set the flag rm.na=T as a second argument, separate arguments by comma)

In [5]:
mean(df$SQ1, na.rm=T)

[1] 24.07572

In [6]:
sd(df$SQ1, na.rm=T)

[1] 3.296855

While you use mean and sd for metric variables, create frequency tables for short scales:

In [7]:
table(df$SQ2)


               female       male non-binary      other 
         9        301        140          4          3 

Is there a missing column identifier, i.e., invisible text in our data? Look up the table headers:

In [8]:
names(table(df$SQ2))

[1] ""           "female"     "male"       "non-binary" "other"     

You should replace empty text values with something more meaningful. Unknown data are always coded NA. Apply this change to the raw data:

In [39]:
df$SQ2[df$SQ2 == ""] = NA
t = table(df$SQ2)
t


    female       male non-binary      other 
       301        140          4          3 

In case you want to include NAs in your frequency table use

In [40]:
tna = table(df$SQ2, useNA = "always")
tna


    female       male non-binary      other       <NA> 
       301        140          4          3          9 

Use prop.table() to calculate the relative frequencies. While ...

In [41]:
prop.table(t)


     female        male  non-binary       other 
0.671875000 0.312500000 0.008928571 0.006696429 

... returns valid relative frequencies, ...

In [12]:
prop.table(tna)


     female        male  non-binary       other        <NA> 
0.658643326 0.306345733 0.008752735 0.006564551 0.019693654 

... returns raw relative frequencies.

Transform the relative frequencies [0...1] into percentages  ...

In [13]:
prop.table(t)*100


    female       male non-binary      other 
67.1875000 31.2500000  0.8928571  0.6696429 

... and round the results to a single decimal unit:

In [14]:
round(prop.table(t)*100,1)


    female       male non-binary      other 
      67.2       31.2        0.9        0.7 

Create an even more condensed version by recoding the small categories into one. If you're a lazy typer, simply look up the relevant headers from the table names ...

In [16]:
names(t)[c(3,4)]

[1] "non-binary" "other"     

Note that c(3,4) can be written as 3:4, too:

In [15]:
names(t)[3:4]

[1] "non-binary" "other"     

There is, however, a fundamental difference between these two definitions. While the colon ":" defines a complete enumeration, e.g.:

In [17]:
3:14

 [1]  3  4  5  6  7  8  9 10 11 12 13 14

 a collection c() can be usesingle seperated values can only be defined with

In [18]:
c(3,14)

[1]  3 14

So take great care when working with colons! 

We define the selection with 

In [19]:
selection = names(t)[c(3,4)]
selection

[1] "non-binary" "other"     

and apply it to our variable in oder to recode values of selected cases in the raw data:

In [20]:
df$SQ2[df$SQ2 %in% selection] = "diverse"
t = table(df$SQ2)
t


diverse  female    male 
      7     301     140 

Note that tables report values in alphabetical order. To force a different behavior, use factors. We'll come back to this later.

For now, we can simply reorder the columns to out preferences:

In [22]:
t[c(2, 3, 1)]


 female    male diverse 
    301     140       7 

### Want to go beyond simple numbers and create tabular output?

#### Compile parameters to tables

Use collections to quickly bind multiple results into a single output:

In [23]:
c(mean(df$SQ1, na.rm=T), sd(df$SQ1, na.rm=T))

[1] 24.075724  3.296855

... or create a dataframe to even include column names:

In [24]:
df_parameters = data.frame(Mean = mean(df$SQ1, na.rm=T), SD = sd(df$SQ1, na.rm=T))
df_parameters

  Mean     SD      
1 24.07572 3.296855

#### Combine parameter tables for multiple variables

To create a table with parameters of multiple variables, we have to include the variable name first ...

In [25]:
df_parameters = data.frame(Variable = "Age", Mean = mean(df$SQ1, na.rm=T), SD = sd(df$SQ1, na.rm=T))
df_parameters

  Variable Mean     SD      
1 Age      24.07572 3.296855

... and add a new row for each additional variable with rbind() (row bind):

In [26]:
newRow = data.frame(Variable = "Working h/week", Mean = mean(df$SQ9, na.rm=T), SD = sd(df$SQ9, na.rm=T))
newRow

  Variable       Mean     SD      
1 Working h/week 16.61582 12.47664

In [27]:
df_parameters = rbind(df_parameters, newRow)
df_parameters

  Variable       Mean     SD       
1 Age            24.07572  3.296855
2 Working h/week 16.61582 12.476636

#### Compile frequency tables 

You can do similar operations with tables. To access only the numeric values in the table use as.numeric():

In [28]:
as.numeric(t)

[1]   7 301 140

In order to see both absolute and relative frequencies, create a dataframe:

In [29]:
df_frequencies = data.frame(Value = names(t), Count = as.numeric(t), Percent= as.numeric(round(prop.table(t)*100, 1)))
df_frequencies

  Value   Count Percent
1 diverse   7    1.6   
2 female  301   67.2   
3 male    140   31.2   

To reorder refer to the table rows now: 

In [31]:
df_frequencies[c(2,3,1),]

  Value   Count Percent
2 female  301   67.2   
3 male    140   31.2   
1 diverse   7    1.6   

See how you can use R to customize output to your individual needs?

#### Combine frequency tables for multiple variables

In order to bind tables for multiple variables we have to assign the variable name, too. We need, however, a single name for several lines of output, i.e., the variable name should be allocated to the first row, while all remaining rows should become empty. In our table with three rows, we have to insert "Gender" into the first row and "" into all remaining 3-1=2 rows. That said, we have to create a collection c("Gender", "", ""). More generally we have to repeat "" (number of rows - 1) times. In R:
    

In [36]:
df_frequencies = data.frame(Variable=c("Gender", rep("", nrow(t)-1)),
                            Value = names(t), Count = as.numeric(t), 
                            Percent= as.numeric(round(prop.table(t)*100, 1)))
df_frequencies

  Variable Value   Count Percent
1 Gender   diverse   7    1.6   
2          female  301   67.2   
3          male    140   31.2   

Now we can add any additional frequency table with rbind() (see above).